In [ ]:
import numpy as np

In [2]:
def american_option(S_ini, K, T, r, u, d, N, opttype):
    dt = T / N  # Define time step
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    C = np.zeros([N + 1, N + 1])  # call prices
    S = np.zeros([N + 1, N + 1])  # underlying price

    for i in range(0, N + 1):
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
        if opttype == "C":
            C[N, i] = max(S[N, i] - K, 0)
        else:
            C[N, i] = max(K - S[N, i], 0)

    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (
                p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i]
            )  # Computing the European option prices
            S[j, i] = (
                S_ini * (u ** (i)) * (d ** (j - i))
            )  # Underlying evolution for each node
            if opttype == "C":
                C[j, i] = max(
                    C[j, i], S[j, i] - K
                )  # Decision between the European option price and the payoff from early-exercise
            else:
                C[j, i] = max(
                    C[j, i], K - S[j, i]
                )  # Decision between the European option price and the payoff from early-exercise

    return C[0, 0], C, S

In [3]:
def american_option_vol(S_ini, K, T, r, sigma, N, opttype):
    dt = T / N  # Define time step
    u = np.exp(sigma * np.sqrt(dt))  # Define u
    d = np.exp(-sigma * np.sqrt(dt))  # Define d
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    C = np.zeros([N + 1, N + 1])  # call prices
    S = np.zeros([N + 1, N + 1])  # underlying price

    for i in range(0, N + 1):
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
        if opttype == "C":
            C[N, i] = max(S[N, i] - K, 0)
        else:
            C[N, i] = max(K - S[N, i], 0)

    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (
                p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i]
            )  # Computing the European option prices
            S[j, i] = (
                S_ini * (u ** (i)) * (d ** (j - i))
            )  # Underlying evolution for each node
            if opttype == "C":
                C[j, i] = max(
                    C[j, i], S[j, i] - K
                )  # Decision between the European option price and the payoff from early-exercise
            else:
                C[j, i] = max(
                    C[j, i], K - S[j, i]
                )  # Decision between the European option price and the payoff from early-exercise

    return C[0, 0], C, S

In [ ]:
S0 = 100, K = 100, r = 5%, T = 0.25, σ = 20%

In [ ]:
american_option_vol(S_ini, K, T, r, sigma, N, opttype)

In [22]:
price_call, C, S = american_option_vol(100, 100, 0.25, 0.05, 0.2, 500, "C")
print(price_call)

4.613000899184725


In [23]:
call_delta_0 = (C[1, 1] - C[1, 0]) / (S[1, 1] - S[1, 0])
call_delta_0

np.float64(0.569426057103601)

In [24]:
price_put, C, S = american_option_vol(100, 100, 0.25, 0.05, 0.2, 500, "P")
print(price_put)

3.4788259769519283


In [26]:
put_delta_0 = (C[1, 1] - C[1, 0]) / (S[1, 1] - S[1, 0])
put_delta_0

np.float64(-0.4495002366123652)

In [1]:
class TrinomialModel(object):  # Here we start defining our 'class' --> Trinomial Model!
    # First, a method to initialize our `TrinomialModel` algorithm!
    def __init__(self, S0, r, sigma, mat):
        self.__s0 = S0
        self.__r = r
        self.__sigma = sigma
        self.__T = mat

    # Second, we build a method (function) to compute the risk-neutral probabilities!
    def __compute_probs(self):
        self.__pu = (
            (
                np.exp(self.__r * self.__h / 2)
                - np.exp(-self.__sigma * np.sqrt(self.__h / 2))
            )
            / (
                np.exp(self.__sigma * np.sqrt(self.__h / 2))
                - np.exp(-self.__sigma * np.sqrt(self.__h / 2))
            )
        ) ** 2
        self.__pd = (
            (
                -np.exp(self.__r * self.__h / 2)
                + np.exp(self.__sigma * np.sqrt(self.__h / 2))
            )
            / (
                np.exp(self.__sigma * np.sqrt(self.__h / 2))
                - np.exp(-self.__sigma * np.sqrt(self.__h / 2))
            )
        ) ** 2
        self.__pm = 1 - self.__pu - self.__pd

        assert 0 <= self.__pu <= 1.0, "p_u should lie in [0, 1] given %s" % self.__pu
        assert 0 <= self.__pd <= 1.0, "p_d should lie in [0, 1] given %s" % self.__pd
        assert 0 <= self.__pm <= 1.0, "p_m should lie in [0, 1] given %s" % self.__pm

    # Third, this method checks whether the given parameters are alright and that we have a 'recombining tree'!
    def __check_up_value(self, up):
        if up is None:
            up = np.exp(self.__sigma * np.sqrt(2 * self.__h))

        assert up > 0.0, "up should be non negative"

        down = 1 / up

        assert down < up, "up <= 1. / up = down"

        self.__up = up
        self.__down = down

    # Four, we use this method to compute underlying stock price path
    def __gen_stock_vec(self, nb):
        vec_u = self.__up * np.ones(nb)
        np.cumprod(vec_u, out=vec_u)

        vec_d = self.__down * np.ones(nb)
        np.cumprod(vec_d, out=vec_d)

        res = np.concatenate((vec_d[::-1], [1.0], vec_u))
        res *= self.__s0

        return res

    # Fifth, we declare a Payoff method to be completed afterwards depending on the instrument we are pricing!
    def payoff(self, stock_vec):
        raise NotImplementedError()

    # Sixth, compute current prices!
    def compute_current_price(self, crt_vec_stock, nxt_vec_prices):
        expectation = np.zeros(crt_vec_stock.size)
        for i in range(expectation.size):
            tmp = nxt_vec_prices[i] * self.__pd
            tmp += nxt_vec_prices[i + 1] * self.__pm
            tmp += nxt_vec_prices[i + 2] * self.__pu

            expectation[i] = tmp

        return self.__discount * expectation

    # Seventh, Option pricing!
    def price(self, nb_steps, up=None):
        assert nb_steps > 0, "nb_steps shoud be > 0"

        nb_steps = int(nb_steps)

        self.__h = self.__T / nb_steps
        self.__check_up_value(up)
        self.__compute_probs()

        self.__discount = np.exp(-self.__r * self.__h)

        final_vec_stock = self.__gen_stock_vec(nb_steps)
        final_payoff = self.payoff(final_vec_stock)
        nxt_vec_prices = final_payoff

        for i in range(1, nb_steps + 1):
            vec_stock = self.__gen_stock_vec(nb_steps - i)
            nxt_vec_prices = self.compute_current_price(vec_stock, nxt_vec_prices)

        return nxt_vec_prices[0]

In [2]:
class TrinomialCall(TrinomialModel):
    def __init__(self, S0, r, sigma, mat, K):
        super(TrinomialCall, self).__init__(S0, r, sigma, mat)
        self.__K = K

    def payoff(self, s):
        return np.maximum(s - self.__K, 0.0)

In [34]:
class TrinomialPut(TrinomialModel):
    def __init__(self, S0, r, sigma, mat, K):
        super(TrinomialPut, self).__init__(S0, r, sigma, mat)
        self.__K = K

    def payoff(self, s):
        return np.maximum(self.__K - s, 0.0)

In [ ]:
strikes = [90, 95, 100, 105, 110]

In [6]:
call_tree_1 = TrinomialCall(100.0, 0.05, 0.2, 0.25, 90.0)

In [12]:
print(call_tree_1.price(500))

11.670327451662752


In [13]:
call_tree_2 = TrinomialCall(100.0, 0.05, 0.2, 0.25, 95.0)

In [14]:
print(call_tree_2.price(500))

7.7141448719694825


In [15]:
call_tree_3 = TrinomialCall(100.0, 0.05, 0.2, 0.25, 100.0)

In [20]:
print(call_tree_3.price(500))

4.6139988891402375


In [21]:
call_tree_4 = TrinomialCall(100.0, 0.05, 0.2, 0.25, 105.0)

In [24]:
print(call_tree_4.price(500))

2.478377884182365


In [25]:
call_tree_5 = TrinomialCall(100.0, 0.05, 0.2, 0.25, 110.0)

In [28]:
print(call_tree_5.price(500))

1.1904782319811602


In [35]:
put_tree_1 = TrinomialPut(100.0, 0.05, 0.2, 0.25, 90)

In [44]:
print(put_tree_1.price(500))

0.5523294961164659


In [48]:
put_tree_2 = TrinomialPut(100.0, 0.05, 0.2, 0.25, 95)

In [49]:
print(put_tree_2.price(500))

1.5340359188927295


In [50]:
put_tree_3 = TrinomialPut(100.0, 0.05, 0.2, 0.25, 100)

In [51]:
print(put_tree_3.price(500))

3.371778938533021


In [52]:
put_tree_4 = TrinomialPut(100.0, 0.05, 0.2, 0.25, 105)

In [53]:
print(put_tree_4.price(500))

6.174046936044686


In [54]:
put_tree_5 = TrinomialPut(100.0, 0.05, 0.2, 0.25, 110)

In [55]:
print(put_tree_5.price(500))

9.824036286313019


In [1]:

import numpy as np

# Model parameters
S0 = 100.0
r = 0.05
T = 0.25
K = 100.0
sigma = 0.20
mu = -0.5
delta = 0.22

# Simulation settings
N = 250
M = 200_000
dt = T / N

# Jump compensator
kappa = np.exp(mu + 0.5 * delta**2) - 1

def merton_mc(lambda_jump, seed=12345):
    rng = np.random.default_rng(seed)

    call_sum = 0.0
    put_sum = 0.0
    batch_size = 20_000

    for start in range(0, M, batch_size):
        m = min(batch_size, M - start)
        logS = np.full(m, np.log(S0))

        for _ in range(N):
            Z = rng.standard_normal(m)

            # Number of jumps per time step
            Nj = rng.poisson(lambda_jump * dt, size=m)

            # Aggregate normally distributed jump sizes
            jump = (
                Nj * mu
                + np.sqrt(Nj) * delta
                * rng.standard_normal(m)
            )

            logS += (
                (r - lambda_jump * kappa
                 - 0.5 * sigma**2) * dt
                + sigma * np.sqrt(dt) * Z
                + jump
            )

        ST = np.exp(logS)

        call_sum += np.maximum(ST - K, 0).sum()
        put_sum += np.maximum(K - ST, 0).sum()

    call_price = np.exp(-r * T) * call_sum / M
    put_price = np.exp(-r * T) * put_sum / M

    return {
        "lambda": lambda_jump,
        "call_price": call_price,
        "put_price": put_price
    }


# Case 1: Lambda = 0.75
result_075 = merton_mc(0.75, seed=12345)

# Case 2: Lambda = 0.25
result_025 = merton_mc(0.25, seed=67890)

print("Lambda = 0.75:", result_075)
print("Lambda = 0.25:", result_025)


Lambda = 0.75: {'lambda': 0.75, 'call_price': np.float64(8.166826141666528), 'put_price': np.float64(6.922095676155614)}
Lambda = 0.25: {'lambda': 0.25, 'call_price': np.float64(5.733238112938215), 'put_price': np.float64(4.485923497729068)}


In [2]:

import numpy as np

# Original model parameters
S0 = 100.0
r = 0.05
T = 0.25
K = 100.0
sigma = 0.20
mu = -0.5
delta = 0.22

# Simulation parameters
N = 250
M = 200_000
dt = T / N
h = 0.50

kappa = np.exp(mu + 0.5 * delta**2) - 1


def merton_greeks(lambda_jump, seed):

    rng = np.random.default_rng(seed)

    # Three underlying stock prices
    spots = np.array([S0-h, S0, S0+h])

    # Call and put payoff accumulators
    totals = np.zeros((3, 2))

    batch_size = 20_000

    for start in range(0, M, batch_size):

        m = min(batch_size, M-start)

        logG = np.zeros(m)

        for _ in range(N):

            Z = rng.standard_normal(m)

            Nj = rng.poisson(lambda_jump * dt, size=m)

            jump = (
                Nj * mu
                + np.sqrt(Nj) * delta
                * rng.standard_normal(m)
            )

            logG += (
                (r - lambda_jump*kappa
                 - 0.5*sigma**2)*dt
                + sigma*np.sqrt(dt)*Z
                + jump
            )

        # Common simulated growth factors
        G = np.exp(logG)

        for j, S in enumerate(spots):

            ST = S * G

            totals[j, 0] += np.maximum(ST-K, 0).sum()
            totals[j, 1] += np.maximum(K-ST, 0).sum()

    # Discounted option prices
    prices = np.exp(-r*T) * totals / M

    # Delta calculations
    delta_call = (
        prices[2, 0] - prices[0, 0]
    ) / (2*h)

    delta_put = (
        prices[2, 1] - prices[0, 1]
    ) / (2*h)

    # Gamma calculations
    gamma_call = (
        prices[2, 0] - 2*prices[1, 0]
        + prices[0, 0]
    ) / h**2

    gamma_put = (
        prices[2, 1] - 2*prices[1, 1]
        + prices[0, 1]
    ) / h**2

    return {
        "prices": prices,
        "call_delta": delta_call,
        "call_gamma": gamma_call,
        "put_delta": delta_put,
        "put_gamma": gamma_put
    }


# Case 1: Lambda = 0.75
greeks_075 = merton_greeks(0.75, seed=12345)

# Case 2: Lambda = 0.25
greeks_025 = merton_greeks(0.25, seed=67890)

print("Lambda = 0.75:", greeks_075)
print("Lambda = 0.25:", greeks_025)


Lambda = 0.75: {'prices': array([[7.80528949, 7.06057158],
       [8.16682614, 6.92209568],
       [8.5345688 , 6.78982579]]), 'call_delta': np.float64(0.729279310984623), 'call_gamma': np.float64(0.024824061132704855), 'put_delta': np.float64(-0.2707457941643643), 'put_gamma': np.float64(0.02482406113271196)}
Lambda = 0.25: {'prices': array([[5.41913421, 4.67184507],
       [5.73323811, 4.4859235 ],
       [6.05603879, 4.3086987 ]]), 'call_delta': np.float64(0.6369045716622157), 'call_gamma': np.float64(0.034787091398932546), 'put_delta': np.float64(-0.36314637498375646), 'put_gamma': np.float64(0.03478709139892544)}


In [3]:
### Question 19

In [1]:

import numpy as np

def price_down_in_put(
    S0=100, K=90, H=90,
    r=0.05, T=0.25,
    sigma=0.20, lam=0.75,
    mu=-0.5, delta=0.22,
    M=200000, N=250, seed=12345
):
    np.random.seed(seed)

    dt = T / N

    # Merton jump compensator
    kappa = np.exp(mu + 0.5 * delta**2) - 1

    drift = (r - lam*kappa - 0.5*sigma**2)*dt

    logS = np.full(M, np.log(S0))
    barrier_hit = np.zeros(M, dtype=bool)

    for t in range(N):
        Z = np.random.standard_normal(M)
        ZJ = np.random.standard_normal(M)

        Nj = np.random.poisson(lam*dt, M)

        diffusion = sigma*np.sqrt(dt)*Z
        jumps = Nj*mu + delta*np.sqrt(Nj)*ZJ

        logS += drift + diffusion + jumps

        # Track whether the barrier has been reached
        barrier_hit |= (logS <= np.log(H))

    ST = np.exp(logS)

    # Down-and-in put payoff
    payoff = np.maximum(K - ST, 0) * barrier_hit

    price = np.exp(-r*T)*np.mean(payoff)

    return {
        "PDI_price": price,
        "barrier_hit_probability": np.mean(barrier_hit),
        "ITM_and_activated_probability": np.mean(
            barrier_hit & (ST < K)
        ),
        "mean_terminal_stock": np.mean(ST)
    }


result = price_down_in_put()

print(result)


{'PDI_price': np.float64(4.4295157828873775), 'barrier_hit_probability': np.float64(0.24493), 'ITM_and_activated_probability': np.float64(0.182625), 'mean_terminal_stock': np.float64(101.26566730949152)}


In [1]:
import numpy as np

In [2]:
def american_option_vol(S_ini, K, T, r, sigma, N, opttype):
    dt = T / N  # Define time step
    u = np.exp(sigma * np.sqrt(dt))  # Define u
    d = np.exp(-sigma * np.sqrt(dt))  # Define d
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    C = np.zeros([N + 1, N + 1])  # call prices
    S = np.zeros([N + 1, N + 1])  # underlying price

    for i in range(0, N + 1):
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
        if opttype == "C":
            C[N, i] = max(S[N, i] - K, 0)
        else:
            C[N, i] = max(K - S[N, i], 0)

    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (
                p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i]
            )  # Computing the European option prices
            S[j, i] = (
                S_ini * (u ** (i)) * (d ** (j - i))
            )  # Underlying evolution for each node
            if opttype == "C":
                C[j, i] = max(
                    C[j, i], S[j, i] - K
                )  # Decision between the European option price and the payoff from early-exercise
            else:
                C[j, i] = max(
                    C[j, i], K - S[j, i]
                )  # Decision between the European option price and the payoff from early-exercise

    return C[0, 0], C, S

In [ ]:
So=100, K=100, r=0.05, sigma=0.20, T=0.25, N=25

In [4]:
price_put, C, S = american_option_vol(100, 100, 0.25, 0.05, 0.2, 25, "P")
print(price_put)

3.5186345979821225


In [7]:

import numpy as np
import pandas as pd

# ==========================================
# 1. MODEL PARAMETERS
# ==========================================

S0 = 100.0
K = 100.0
r = 0.05
T = 0.25
N = 25
sigma = 0.20

dt = T / N

u = np.exp(sigma * np.sqrt(dt))
d = 1 / u

p = (np.exp(r * dt) - d) / (u - d)

print("BINOMIAL MODEL PARAMETERS")
print(f"u = {u:.8f}")
print(f"d = {d:.8f}")
print(f"Risk-neutral probability = {p:.8f}")


# ==========================================
# 2. INITIALIZE MATRICES
# ==========================================

S = np.zeros((N + 1, N + 1))
P = np.zeros((N + 1, N + 1))
Delta = np.full((N, N), np.nan)


# ==========================================
# 3. STOCK PRICES AND TERMINAL PAYOFFS
# ==========================================

for i in range(N + 1):

    S[N, i] = S0 * (u ** i) * (d ** (N - i))

    P[N, i] = max(K - S[N, i], 0)


# ==========================================
# 4. BACKWARD INDUCTION
# AMERICAN PUT VALUATION
# ==========================================

for j in range(N - 1, -1, -1):

    for i in range(j + 1):

        S[j, i] = S0 * (u ** i) * (d ** (j - i))

        continuation = np.exp(-r * dt) * (
            p * P[j + 1, i + 1]
            + (1 - p) * P[j + 1, i]
        )

        exercise = max(K - S[j, i], 0)

        P[j, i] = max(continuation, exercise)

        # Delta at each node
        Delta[j, i] = (
            P[j + 1, i + 1] - P[j + 1, i]
        ) / (
            S[j + 1, i + 1] - S[j + 1, i]
        )


# ==========================================
# 5. INITIAL OPTION PRICE AND HEDGE
# ==========================================

print("\nAMERICAN PUT RESULTS")
print(f"American Put Price = {P[0, 0]:.6f}")
print(f"Initial Delta = {Delta[0, 0]:.6f}")

print("\nINITIAL SELLER HEDGE")
print(f"Short {-Delta[0, 0]:.6f} shares")


# ==========================================
# 6. DELTA HEDGING AT EVERY NODE
# ==========================================

results = []

for j in range(N):

    for i in range(j + 1):

        current_delta = Delta[j, i]

        if j == 0:

            trade = current_delta

            previous_position = 0.0

        else:

            # Determine the previous node based on
            # the incoming path.

            previous_position = None

            if i <= j - 1:

                down_parent = Delta[j - 1, i]

                trade_down = current_delta - down_parent

            else:

                trade_down = None

            if i >= 1:

                up_parent = Delta[j - 1, i - 1]

                trade_up = current_delta - up_parent

            else:

                trade_up = None

            trade = None

        # Record the node information
        results.append({
            "Step": j,
            "Up_Moves": i,
            "Down_Moves": j - i,
            "Stock_Price": S[j, i],
            "Put_Value": P[j, i],
            "Put_Delta": current_delta,
            "Down_Parent_Trade": (
                trade_down if j > 0 else np.nan
            ),
            "Up_Parent_Trade": (
                trade_up if j > 0 else np.nan
            ),
            "Initial_Hedge": (
                trade if j == 0 else np.nan
            )
        })


# ==========================================
# 7. DISPLAY RESULTS
# ==========================================

df = pd.DataFrame(results)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\nCOMPLETE DELTA HEDGING TABLE")

print(df.round(6).to_string(index=False))


# ==========================================
# 8. EXPORT TO EXCEL AND CSV
# ==========================================

df.to_csv("american_put_delta_hedging.csv", index=False)

df.to_excel(
    "american_put_delta_hedging.xlsx",
    index=False
)

print("\nFiles successfully generated:")
print("1. american_put_delta_hedging.csv")
print("2. american_put_delta_hedging.xlsx")


# ==========================================
# 9. DISPLAY DELTA TREE
# ==========================================

print("\nDELTA TREE")

for j in range(N):

    print(f"\nStep {j}:")

    for i in range(j + 1):

        print(
            f"Node ({j},{i}): "
            f"Stock = {S[j,i]:.4f}, "
            f"Put Delta = {Delta[j,i]:.6f}"
        )


BINOMIAL MODEL PARAMETERS
u = 1.02020134
d = 0.98019867
Risk-neutral probability = 0.50750246

AMERICAN PUT RESULTS
American Put Price = 3.518635
Initial Delta = -0.450220

INITIAL SELLER HEDGE
Short 0.450220 shares

COMPLETE DELTA HEDGING TABLE
 Step  Up_Moves  Down_Moves  Stock_Price  Put_Value  Put_Delta  Down_Parent_Trade  Up_Parent_Trade  Initial_Hedge
    0         0           0   100.000000   3.518635  -0.450220                NaN              NaN       -0.45022
    1         0           1    98.019867   4.434406  -0.538376          -0.088156              NaN            NaN
    1         1           0   102.020134   2.633406  -0.368024                NaN         0.082195            NaN
    2         0           2    96.078944   5.507964  -0.630891          -0.092515              NaN            NaN
    2         1           1   100.000000   3.396960  -0.452117          -0.084093         0.086259            NaN
    2         2           0   104.081077   1.895023  -0.289618        

In [8]:

# ==========================================
# CASH ACCOUNT EVOLUTION: ALL-UP PATH
# ==========================================

cash = P[0, 0] - Delta[0, 0] * S0

previous_delta = Delta[0, 0]

cash_results = []

# Initial position
cash_results.append({
    "Step": 0,
    "Movement": "Initial",
    "Stock_Price": S[0, 0],
    "Put_Delta": Delta[0, 0],
    "Shares_Traded": Delta[0, 0],
    "Cash_Interest": 0.0,
    "Cash_Account": cash
})

# Follow 25 consecutive upward movements
for j in range(1, N):

    i = j  # All-up path

    stock_price = S[j, i]

    current_delta = Delta[j, i]

    # Accrue interest on cash account
    interest = cash * (np.exp(r * dt) - 1)

    cash *= np.exp(r * dt)

    # Rebalance stock hedge
    shares_traded = current_delta - previous_delta

    cash -= shares_traded * stock_price

    # Record results
    cash_results.append({
        "Step": j,
        "Movement": "Up",
        "Stock_Price": stock_price,
        "Put_Delta": current_delta,
        "Shares_Traded": shares_traded,
        "Cash_Interest": interest,
        "Cash_Account": cash
    })

    previous_delta = current_delta


# ==========================================
# MATURITY: STEP 25
# ==========================================

stock_price = S[N, N]

# Accrue final period interest
interest = cash * (np.exp(r * dt) - 1)

cash *= np.exp(r * dt)

# Close stock hedge
shares_traded = -previous_delta

cash -= shares_traded * stock_price

# Settle American put payoff
payoff = max(K - stock_price, 0)

cash -= payoff

cash_results.append({
    "Step": N,
    "Movement": "Up (Maturity)",
    "Stock_Price": stock_price,
    "Put_Delta": np.nan,
    "Shares_Traded": shares_traded,
    "Cash_Interest": interest,
    "Cash_Account": cash
})


# ==========================================
# DISPLAY CASH ACCOUNT TABLE
# ==========================================

cash_df = pd.DataFrame(cash_results)

print("\nCASH ACCOUNT EVOLUTION")
print("Path: UP, UP, UP, ... UP")

print(cash_df.round(6).to_string(index=False))

print("\nFINAL RESULTS")
print(f"Final Stock Price: {stock_price:.6f}")
print(f"Final Put Payoff: {payoff:.6f}")
print(f"Final Cash Account: {cash:.6f}")


# Export results
cash_df.to_excel(
    "american_put_cash_account.xlsx",
    index=False
)

print("\nCash account saved to american_put_cash_account.xlsx")



CASH ACCOUNT EVOLUTION
Path: UP, UP, UP, ... UP
 Step      Movement  Stock_Price  Put_Delta  Shares_Traded  Cash_Interest  Cash_Account
    0       Initial   100.000000  -0.450220      -0.450220       0.000000     48.540630
    1            Up   102.020134  -0.368024       0.082195       0.024276     40.179313
    2            Up   104.081077  -0.289618       0.078407       0.020095     32.038769
    3            Up   106.183655  -0.217877       0.071741       0.016023     24.437038
    4            Up   108.328707  -0.155327       0.062550       0.012222     17.673297
    5            Up   110.517092  -0.103795       0.051532       0.008839     11.986964
    6            Up   112.749685  -0.064107       0.039687       0.005995      7.518237
    7            Up   115.027380  -0.035923       0.028185       0.003760      4.279987
    8            Up   117.351087  -0.017796       0.018127       0.002141      2.154893
    9            Up   119.721736  -0.007501       0.010294       0.0010